# Quality Gate — Silver Layer

This notebook runs comprehensive data quality checks on `workspace.silver.flights_clean` before promoting data to the Gold layer. It validates row counts, critical column completeness, duplicate records, domain value validity, and airport code completeness. The result is passed downstream via `dbutils.jobs.taskValues` so a Job condition task can decide whether to proceed.

In [0]:
from pyspark.sql import functions as F

# ── Read silver layer ──
df = spark.sql("SELECT * FROM workspace.silver.flights_clean")
total_rows = df.count()

results = []  # list of (check_name, passed, detail)

def add_check(name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {name}: {detail}")
    results.append((name, passed))

# ── Check 1: Row count > 0 ──
add_check("row_count_gt_zero", total_rows > 0, f"total_rows={total_rows}")

# ── Check 2: Critical columns have zero NULLs ──
critical_cols = ["flight_number", "flight_type", "status"]
for col_name in critical_cols:
    missing = df.filter(F.col(col_name).isNull()).count()
    pct = round(missing * 100 / total_rows, 2) if total_rows else 0
    add_check(f"no_nulls_{col_name}", missing == 0, f"missing={missing} ({pct}%)")

# ── Check 3: flight_type values are valid ──
valid_flight_types = ["departure", "arrival"]
invalid_type_count = df.filter(~F.col("flight_type").isin(valid_flight_types)).count()
add_check("valid_flight_type", invalid_type_count == 0,
          f"invalid_rows={invalid_type_count} (expected: departure/arrival)")

# ── Check 4: Duplicate flight records ──
# A duplicate = same flight_number + movement.scheduledTime.utc + flight_type
dupes = (
    df.groupBy("flight_number", "`movement.scheduledTime.utc`", "flight_type")
      .agg(F.count("*").alias("cnt"))
      .filter(F.col("cnt") > 1)
)
dupe_groups = dupes.count()
dupe_rows = dupes.agg(F.sum("cnt")).collect()[0][0] or 0
add_check("no_duplicate_flights", dupe_groups == 0,
          f"duplicate_groups={dupe_groups}, extra_rows={dupe_rows - dupe_groups if dupe_groups else 0}")

# ── Check 5: Unknown status threshold (< 90%) ──
unknown_status = df.filter(F.col("status") == "Unknown").count()
unknown_pct = round(unknown_status * 100 / total_rows, 2) if total_rows else 0
add_check("unknown_status_below_90pct", unknown_pct < 90,
          f"unknown_rows={unknown_status} ({unknown_pct}%)")

# ── Check 6: Airport code completeness ──
# Departures should have origin_airport_iata; arrivals should have destination_airport_iata
dep_missing_origin = df.filter(
    (F.col("flight_type") == "departure") & F.col("origin_airport_iata").isNull()
).count()
arr_missing_dest = df.filter(
    (F.col("flight_type") == "arrival") & F.col("destination_airport_iata").isNull()
).count()
add_check("departures_have_origin", dep_missing_origin == 0,
          f"departures_missing_origin={dep_missing_origin}")
add_check("arrivals_have_destination", arr_missing_dest == 0,
          f"arrivals_missing_destination={arr_missing_dest}")

# ── Check 7: Non-critical column NULL thresholds ──
# These columns are allowed to have NULLs but should not exceed a threshold
threshold_checks = [
    ("airline.name", 5.0),
    ("airline.iata", 5.0),
    ("airline.icao", 5.0),
    ("movement.terminal", 50.0),
]
for col_name, threshold in threshold_checks:
    missing = df.filter(F.col(f"`{col_name}`").isNull()).count()
    pct = round(missing * 100 / total_rows, 2) if total_rows else 0
    add_check(f"null_threshold_{col_name.replace('.', '_')}", pct <= threshold,
              f"missing={missing} ({pct}%, threshold={threshold}%)")

# ── Summary ──
passed = sum(1 for _, p in results if p)
failed = sum(1 for _, p in results if not p)
all_passed = failed == 0

print(f"\n{'='*50}")
print(f"Quality Gate Summary: {passed} passed, {failed} failed")
print(f"Overall result: {'PASS' if all_passed else 'FAIL'}")
print(f"{'='*50}")

In [0]:
# Set task value for the condition task to evaluate
if all_passed:
    print("Quality gate PASSED - proceeding to gold layer")
    dbutils.jobs.taskValues.set(key="quality_status", value="pass")
else:
    failed_checks = [name for name, passed in results if not passed]
    print(f"Quality gate FAILED - issues: {failed_checks}")
    dbutils.jobs.taskValues.set(key="quality_status", value="fail")
    dbutils.jobs.taskValues.set(key="failed_checks", value=failed_checks)

dbutils.notebook.exit("done")